 <h1>

 Airbnb Price Prediction Project

 </h1>

 <h3>

 1) Problem Statement

 </h3>

 <p>

 The goal of this project is to predict Airbnb listing prices using structured listing data such as

 location, room type, review scores, host information, and availability.

 This notebook demonstrates a full machine learning workflow, including data loading, exploratory

 analysis, data cleaning, feature engineering, model building, hyperparameter tuning, evaluation,

 and interpretation.

 </p>

In [ ]:
# =========================================
# 1. IMPORTS
# =========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")


 <h3>

 2) Data Loading

 </h3>

 <p>

 In this section, multiple Airbnb listing datasets are loaded and combined into one dataframe.

 A city label is added to each dataset so the model can learn pricing differences across markets.

 </p>

In [ ]:
# =========================================
# 2. LOAD AND COMBINE MULTIPLE CITY DATASETS
# =========================================
columbus = pd.read_csv("listings.csv/listings - Columbus, Ohio.csv")
columbus["city"] = "Columbus"

nashville = pd.read_csv("listings.csv/listings - Nashville, Tennessee.csv")
nashville["city"] = "Nashville"

new_york = pd.read_csv("listings.csv/listings - New York, New York.csv")
new_york["city"] = "New York"

prague = pd.read_csv("listings.csv/listings - Prague, Czech Republic.csv")
prague["city"] = "Prague"

df = pd.concat([columbus, nashville, new_york, prague], ignore_index=True)

print("Combined dataset shape:", df.shape)
print("\nRows by city:")
print(df["city"].value_counts())
print("\nFirst 5 rows:")
print(df.head())


 <h3>

 3) Data Exploration

 </h3>

 <p>

 This section explores the structure of the data, the amount of missing values, and the distribution

 of prices. The price variable is expected to be right-skewed, so the plots help justify outlier

 removal and log transformation.

 </p>

In [ ]:
# =========================================
# 3. DATA EXPLORATION
# =========================================
print("Dataset shape:", df.shape)

print("\nTop missing values:")
print(df.isnull().sum().sort_values(ascending=False).head(20))

# Clean price for exploration
df["price"] = df["price"].replace(r"[\$,]", "", regex=True)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

print("\nPrice summary statistics:")
print(df["price"].describe())

print("\nTop 10 most expensive listings:")
print(df["price"].sort_values(ascending=False).head(10))

# Raw price histogram
plt.figure(figsize=(10, 5))
plt.hist(df["price"].dropna(), bins=50, edgecolor="black")
plt.xlabel("Price in Dollars")
plt.ylabel("Number of Listings")
plt.title("Raw Airbnb Price Distribution")
plt.tight_layout()
plt.show()

# Histogram up to 95th percentile
price_95 = df["price"].quantile(0.95)

plt.figure(figsize=(10, 5))
plt.hist(df.loc[df["price"] <= price_95, "price"].dropna(), bins=40, edgecolor="black")
plt.xlabel("Price in Dollars")
plt.ylabel("Number of Listings")
plt.title("Airbnb Price Distribution up to the 95th Percentile")
plt.tight_layout()
plt.show()

# Log-price histogram
df["log_price"] = np.log1p(df["price"])

plt.figure(figsize=(10, 5))
plt.hist(df["log_price"].dropna(), bins=40, edgecolor="black")
plt.xlabel("Log Price")
plt.ylabel("Number of Listings")
plt.title("Distribution of Log-Transformed Airbnb Prices")
plt.tight_layout()
plt.show()

# Listings by city
plt.figure(figsize=(8, 5))
df["city"].value_counts().plot(kind="bar")
plt.xlabel("City")
plt.ylabel("Number of Listings")
plt.title("Number of Listings by City")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Listings by room type
if "room_type" in df.columns:
    plt.figure(figsize=(8, 5))
    df["room_type"].value_counts().plot(kind="bar")
    plt.xlabel("Room Type")
    plt.ylabel("Number of Listings")
    plt.title("Number of Listings by Room Type")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Average price by city
city_avg_price = df.groupby("city")["price"].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
city_avg_price.plot(kind="bar")
plt.xlabel("City")
plt.ylabel("Average Price")
plt.title("Average Airbnb Price by City")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Correlation heatmap
candidate_cols = [
    "price",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "number_of_reviews",
    "review_scores_rating",
    "availability_365"
]
existing_cols = [col for col in candidate_cols if col in df.columns]

if len(existing_cols) > 1:
    plt.figure(figsize=(8, 6))
    corr = df[existing_cols].corr(numeric_only=True)
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation Heatmap of Key Numeric Features")
    plt.tight_layout()
    plt.show()


 <h3>

 4) Data Cleaning

 </h3>

 <p>

 The price column is cleaned and extreme outliers are removed. Listings above the 95th percentile

 are excluded to reduce the influence of unusually expensive properties. The target variable is then

 log-transformed to improve model stability.

 </p>

In [ ]:
# =========================================
# 4. CLEAN TARGET VARIABLE
# =========================================
df = df.dropna(subset=["price"]).copy()

upper_limit = df["price"].quantile(0.95)
df = df[df["price"] < upper_limit].copy()

print("Dataset shape after 95th percentile cutoff:", df.shape)
print("Price cutoff used:", upper_limit)

df["log_price"] = np.log1p(df["price"])

plt.figure(figsize=(10, 5))
plt.hist(df["price"].dropna(), bins=40, edgecolor="black")
plt.xlabel("Price")
plt.ylabel("Count")
plt.title("Cleaned Price Distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(df["log_price"].dropna(), bins=40, edgecolor="black")
plt.xlabel("Log Price")
plt.ylabel("Count")
plt.title("Log-Transformed Price Distribution")
plt.tight_layout()
plt.show()


 <h3>

 5) Feature Engineering

 </h3>

 <p>

 New features are created to improve model performance. These include datetime-based features,

 ratio features, pricing features, and location-based clustering features that help capture

 neighborhood-level patterns.

 </p>

In [ ]:
# =========================================
# 5. DATETIME FEATURES
# =========================================
date_cols = ["host_since", "first_review", "last_review", "last_scraped", "calendar_last_scraped"]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

reference_date = df["last_scraped"].max() if "last_scraped" in df.columns else pd.Timestamp.today()

if "host_since" in df.columns:
    df["host_tenure_days"] = (reference_date - df["host_since"]).dt.days

if "first_review" in df.columns:
    df["days_since_first_review"] = (reference_date - df["first_review"]).dt.days

if "last_review" in df.columns:
    df["days_since_last_review"] = (reference_date - df["last_review"]).dt.days


In [ ]:
# =========================================
# 6. FEATURE ENGINEERING
# =========================================

# Convert percentage columns
for col in ["host_response_rate", "host_acceptance_rate"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace("%", "", regex=False)
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Convert boolean-style columns
bool_map = {"t": 1, "f": 0, True: 1, False: 0}
for col in [
    "instant_bookable",
    "host_is_superhost",
    "host_identity_verified",
    "host_has_profile_pic",
    "has_availability"
]:
    if col in df.columns:
        df[col] = df[col].map(bool_map)

# Extract numeric bathrooms from text
if "bathrooms_text" in df.columns:
    df["bathrooms_text_num"] = (
        df["bathrooms_text"]
        .astype(str)
        .str.extract(r"(\d+\.?\d*)")[0]
        .astype(float)
    )

# Count amenities
if "amenities" in df.columns:
    def count_amenities(x):
        try:
            return len(ast.literal_eval(x))
        except Exception:
            return np.nan

    df["amenities_count"] = df["amenities"].apply(count_amenities)

# Distance from dataset center
if "latitude" in df.columns and "longitude" in df.columns:
    df["distance_from_center"] = (
        (df["latitude"] - df["latitude"].mean()) ** 2 +
        (df["longitude"] - df["longitude"].mean()) ** 2
    )

# Stronger engineered features
if "accommodates" in df.columns:
    df["price_per_person"] = df["price"] / (df["accommodates"] + 1)

if "bedrooms" in df.columns and "accommodates" in df.columns:
    df["bedroom_density"] = df["bedrooms"] / (df["accommodates"] + 1)

if "beds" in df.columns and "accommodates" in df.columns:
    df["bed_density"] = df["beds"] / (df["accommodates"] + 1)

if "availability_365" in df.columns:
    df["availability_ratio"] = df["availability_365"] / 365

if "number_of_reviews" in df.columns and "host_listings_count" in df.columns:
    df["reviews_per_listing"] = df["number_of_reviews"] / (df["host_listings_count"] + 1)

if "number_of_reviews" in df.columns and "host_tenure_days" in df.columns:
    df["reviews_per_day"] = df["number_of_reviews"] / (df["host_tenure_days"] + 1)

if "city" in df.columns:
    df["city_avg_price"] = df.groupby("city")["price"].transform("mean")

if "accommodates" in df.columns and "bedrooms" in df.columns:
    df["accommodates_x_bedrooms"] = df["accommodates"] * df["bedrooms"]

# Per-city location clustering
if "latitude" in df.columns and "longitude" in df.columns and "city" in df.columns:
    df["location_cluster"] = -1
    cluster_offset = 0

    for city in df["city"].unique():
        subset = df[df["city"] == city]

        kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
        labels = kmeans.fit_predict(subset[["latitude", "longitude"]])

        df.loc[subset.index, "location_cluster"] = labels + cluster_offset
        cluster_offset += 10

    df["cluster_avg_price"] = df.groupby("location_cluster")["price"].transform("mean")

# Drop completely empty columns to avoid imputation warnings
df = df.dropna(axis=1, how="all")


 <h3>

 6) Train / Validation / Test Split

 </h3>

 <p>

 The data is split into training, validation, and test sets using a 60/20/20 split. Stratification

 by city ensures a balanced distribution of locations across all splits.

 </p>

In [ ]:
# =========================================
# 7. SELECT FEATURES
# =========================================
feature_cols = [
    "city",
    "accommodates",
    "bathrooms",
    "bathrooms_text_num",
    "bedrooms",
    "beds",
    "minimum_nights",
    "maximum_nights",
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "reviews_per_month",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "host_listings_count",
    "host_total_listings_count",
    "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "availability_eoy",
    "estimated_occupancy_l365d",
    "host_response_rate",
    "host_acceptance_rate",
    "latitude",
    "longitude",
    "distance_from_center",
    "host_tenure_days",
    "days_since_first_review",
    "days_since_last_review",
    "amenities_count",
    "instant_bookable",
    "host_is_superhost",
    "host_identity_verified",
    "host_has_profile_pic",
    "has_availability",
    "room_type",
    "property_type",
    "neighbourhood_cleansed",
    "host_response_time",
    "source",
    "price_per_person",
    "bedroom_density",
    "bed_density",
    "availability_ratio",
    "reviews_per_listing",
    "reviews_per_day",
    "city_avg_price",
    "accommodates_x_bedrooms",
    "location_cluster",
    "cluster_avg_price"
]

feature_cols = [col for col in feature_cols if col in df.columns]

X = df[feature_cols].copy()
y = df["log_price"]

print("Number of selected features:", len(feature_cols))
print(feature_cols)


In [ ]:
# =========================================
# 8. TRAIN / DEV / TEST SPLIT
# =========================================
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=X["city"] if "city" in X.columns else None
)

X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=X_train_full["city"] if "city" in X_train_full.columns else None
)

print("Train shape:", X_train.shape)
print("Dev shape:", X_dev.shape)
print("Test shape:", X_test.shape)

if "city" in X.columns:
    print("\nCity distribution in Train:")
    print(X_train["city"].value_counts(normalize=True))
    print("\nCity distribution in Dev:")
    print(X_dev["city"].value_counts(normalize=True))
    print("\nCity distribution in Test:")
    print(X_test["city"].value_counts(normalize=True))


 <h3>

 7) Modeling

 </h3>

 <p>

 Multiple models are compared, including a simple baseline, Ridge Regression, Random Forest, and

 Gradient Boosting. This helps determine whether more flexible models outperform simpler ones.

 </p>

In [ ]:
# =========================================
# 9. PREPROCESSING PIPELINE
# =========================================
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [col for col in X.columns if col not in numeric_cols]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])


In [ ]:
# =========================================
# 10. BASELINE AND MULTIPLE MODELS
# =========================================
baseline_pred_log = np.full(shape=len(y_dev), fill_value=y_train.mean())
baseline_mae_dollars = mean_absolute_error(np.expm1(y_dev), np.expm1(baseline_pred_log))
print("Baseline MAE in dollars:", baseline_mae_dollars)

models = {
    "Ridge": Ridge(),
    "RandomForest": RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
        n_estimators=300,
        max_depth=25,
        min_samples_split=5,
        min_samples_leaf=2
    ),
    "GradientBoosting": GradientBoostingRegressor(
        random_state=42,
        n_estimators=400,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8
    )
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_dev)

    mae_log = mean_absolute_error(y_dev, preds)
    mae_dollars = mean_absolute_error(np.expm1(y_dev), np.expm1(preds))
    r2 = r2_score(y_dev, preds)

    results.append({
        "Model": name,
        "MAE_log": mae_log,
        "MAE_dollars": mae_dollars,
        "R2": r2
    })

results_df = pd.DataFrame(results).sort_values("MAE_dollars")
print(results_df)


 <h3>

 8) Hyperparameter Tuning

 </h3>

 <p>

 The strongest tree-based models are tuned using RandomizedSearchCV to improve performance while

 keeping the search process computationally manageable.

 </p>

In [ ]:
# =========================================
# 11. RANDOM FOREST HYPERPARAMETER TUNING
# =========================================
rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

rf_param_dist = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [15, 20, 25, 30, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None]
}

rf_search = RandomizedSearchCV(
    rf_pipe,
    param_distributions=rf_param_dist,
    n_iter=12,
    cv=3,
    scoring="neg_mean_absolute_error",
    verbose=1,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train_full, y_train_full)

print("Best Random Forest params:")
print(rf_search.best_params_)

best_rf_model = rf_search.best_estimator_


In [ ]:
# =========================================
# 12. XGBOOST MODEL
# =========================================
xgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        n_estimators=500,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1
    ))
])

xgb_pipe.fit(X_train, y_train)

xgb_dev_preds = xgb_pipe.predict(X_dev)

xgb_dev_mae_log = mean_absolute_error(y_dev, xgb_dev_preds)
xgb_dev_mae_dollars = mean_absolute_error(np.expm1(y_dev), np.expm1(xgb_dev_preds))
xgb_dev_r2 = r2_score(y_dev, xgb_dev_preds)

print("XGBoost Dev Results")
print("MAE (log):", xgb_dev_mae_log)
print("MAE (dollars):", xgb_dev_mae_dollars)
print("R2:", xgb_dev_r2)


In [ ]:
# =========================================
# 13. XGBOOST HYPERPARAMETER TUNING
# =========================================
xgb_tune_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_param_dist = {
    "model__n_estimators": [300, 500, 700],
    "model__learning_rate": [0.01, 0.03, 0.05],
    "model__max_depth": [4, 5, 6, 8],
    "model__min_child_weight": [1, 3, 5],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__reg_alpha": [0, 0.1, 0.5],
    "model__reg_lambda": [1.0, 1.5, 2.0]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_tune_pipe,
    param_distributions=xgb_param_dist,
    n_iter=15,
    cv=3,
    scoring="neg_mean_absolute_error",
    verbose=1,
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X_train_full, y_train_full)

print("Best XGBoost params:")
print(xgb_search.best_params_)

best_xgb_model = xgb_search.best_estimator_


 <h3>

 9) Model Evaluation

 </h3>

 <p>

 Final models are evaluated on the held-out test set using MAE and R². Because the target was

 log-transformed during training, MAE is also converted back into dollar units for interpretation.

 </p>

In [ ]:
# =========================================
# 14. FINAL TEST EVALUATION FOR RANDOM FOREST
# =========================================
rf_test_preds = best_rf_model.predict(X_test)

rf_test_mae_log = mean_absolute_error(y_test, rf_test_preds)
rf_test_mae_dollars = mean_absolute_error(np.expm1(y_test), np.expm1(rf_test_preds))
rf_test_r2 = r2_score(y_test, rf_test_preds)

print("\nFinal Random Forest Test Results")
print("MAE (log):", rf_test_mae_log)
print("MAE (dollars):", rf_test_mae_dollars)
print("R2:", rf_test_r2)


In [ ]:
# =========================================
# 15. FINAL TEST EVALUATION FOR XGBOOST
# =========================================
xgb_test_preds = best_xgb_model.predict(X_test)

xgb_test_mae_log = mean_absolute_error(y_test, xgb_test_preds)
xgb_test_mae_dollars = mean_absolute_error(np.expm1(y_test), np.expm1(xgb_test_preds))
xgb_test_r2 = r2_score(y_test, xgb_test_preds)

print("\nFinal XGBoost Test Results")
print("MAE (log):", xgb_test_mae_log)
print("MAE (dollars):", xgb_test_mae_dollars)
print("R2:", xgb_test_r2)


In [ ]:
# =========================================
# 16. ENSEMBLE MODEL
# =========================================
rf_preds = best_rf_model.predict(X_test)
xgb_preds = best_xgb_model.predict(X_test)

ensemble_preds = (rf_preds * 0.5) + (xgb_preds * 0.5)

ensemble_mae = mean_absolute_error(
    np.expm1(y_test),
    np.expm1(ensemble_preds)
)

ensemble_r2 = r2_score(y_test, ensemble_preds)

print("\nEnsemble Results")
print("MAE (dollars):", ensemble_mae)
print("R2:", ensemble_r2)


In [ ]:
# =========================================
# 17. RELATIVE ERROR
# =========================================
mean_price = np.expm1(y_test).mean()
relative_error = ensemble_mae / mean_price

print("Mean price:", mean_price)
print("Relative error:", relative_error)


 <h3>

 Prediction Plots

 </h3>

 <p>

 These plots compare actual and predicted prices for the final Random Forest and XGBoost models.

 A tighter pattern along the diagonal indicates better predictive performance.

 </p>

In [ ]:
# =========================================
# 18. ACTUAL VS PREDICTED FOR RANDOM FOREST
# =========================================
actual_prices = np.expm1(y_test)
rf_predicted_prices = np.expm1(rf_test_preds)

plt.figure(figsize=(8, 6))
plt.scatter(actual_prices, rf_predicted_prices, alpha=0.5)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Airbnb Prices - Random Forest")
plt.tight_layout()
plt.show()


In [ ]:
# =========================================
# 19. ACTUAL VS PREDICTED FOR XGBOOST
# =========================================
xgb_predicted_prices = np.expm1(xgb_test_preds)

plt.figure(figsize=(8, 6))
plt.scatter(actual_prices, xgb_predicted_prices, alpha=0.5)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Airbnb Prices - XGBoost")
plt.tight_layout()
plt.show()


 <h3>

 10) Model Interpretation

 </h3>

 <p>

 Permutation importance is used to identify which original features most influence model performance.

 This helps explain the model and highlights the strongest drivers of Airbnb pricing.

 </p>

In [ ]:
# =========================================
# 20. PERMUTATION IMPORTANCE FOR RANDOM FOREST
# =========================================
perm = permutation_importance(
    best_rf_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

print("\nTop 15 Important Features:")
print(importance_df.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(15), x="importance", y="feature")
plt.title("Top 15 Permutation Importances")
plt.tight_layout()
plt.show()


 <h3>

 11) Conclusion

 </h3>

 <p>

 The final ensemble model achieved the best performance, with a Mean Absolute Error of approximately

 $291 and an R² of about 0.93.

 </p>

 <p>

 This represents a substantial improvement over the baseline model, which had an MAE of about $1087.

 The large reduction in error shows that the selected features and machine learning pipeline capture

 meaningful patterns in Airbnb pricing.

 </p>

 <p>

 Feature importance analysis showed that location-based variables such as latitude, longitude, city,

 and distance from the center were among the strongest predictors. Listing characteristics such as

 room type and engineered features like price per person also played important roles.

 </p>

 <p>

 Additional experiments with Random Forest, XGBoost, and ensembling produced only small improvements

 at the final stage, suggesting that the model is approaching the practical performance limit for

 this dataset.

 </p>

 <p>

 Overall, this project demonstrates that machine learning methods can predict Airbnb prices with

 strong accuracy across multiple cities. Future work could include calendar data, seasonal demand

 features, or textual review information to further improve performance.

 </p>